In [ ]:
import pandas as pd

# Path to your CSV
path = "/content/first_1.3m_rows.csv"  # adjust if needed

# Load CSV
df = pd.read_csv(path)

# Convert timestamp string to datetime (NO unit!)
df["datetime"] = pd.to_datetime(df["timestamp"], utc=True)

# Keep only rows from year 2009
df_2009 = df[df["datetime"].dt.year == 2009]

# Drop helper column
df_2009 = df_2009.drop(columns=["datetime"])

# Save result
df_2009.to_csv("first_1.3m_rows_2009.csv", index=False)

print("Rows before:", len(df))
print("Rows in 2009:", len(df_2009))



Rows before: 1300000
Rows in 2009: 160372


In [ ]:
df_2009["user_id"].nunique()

sorted(df_2009["user_id"].unique())

['user_000001',
 'user_000002',
 'user_000003',
 'user_000004',
 'user_000005',
 'user_000006',
 'user_000007',
 'user_000008',
 'user_000009',
 'user_000010',
 'user_000011',
 'user_000012',
 'user_000013',
 'user_000014',
 'user_000015',
 'user_000016',
 'user_000017',
 'user_000018',
 'user_000019',
 'user_000020',
 'user_000021',
 'user_000022',
 'user_000023',
 'user_000024',
 'user_000025',
 'user_000026',
 'user_000027',
 'user_000028',
 'user_000029',
 'user_000030',
 'user_000031',
 'user_000032',
 'user_000033',
 'user_000034',
 'user_000035',
 'user_000036',
 'user_000037',
 'user_000038',
 'user_000039',
 'user_000040',
 'user_000041',
 'user_000043',
 'user_000044',
 'user_000045',
 'user_000046',
 'user_000047',
 'user_000048',
 'user_000049',
 'user_000050',
 'user_000051',
 'user_000052',
 'user_000053',
 'user_000054',
 'user_000055',
 'user_000056',
 'user_000058',
 'user_000059',
 'user_000060',
 'user_000063',
 'user_000064']

In [ ]:
import pandas as pd
import re
import unicodedata

# Allowed characters:
# - ASCII letters/digits/punctuation
# - Latin-1 Supplement + Latin Extended blocks for accented letters
# - spaces
ALLOWED_RE = re.compile(r"^[\u0009\u000A\u000D\u0020-\u007E\u00A0-\u024F]+$")

def is_clean_latin_text(s, min_alpha=2):
    if not isinstance(s, str):
        return False

    s = unicodedata.normalize("NFKC", s).strip()
    if not s:
        return False

    # Reject if any Hebrew char appears
    if re.search(r"[\u0590-\u05FF]", s):
        return False

    # Reject if contains control / private-use / surrogate categories
    for ch in s:
        if unicodedata.category(ch).startswith("C"):
            return False

    # Reject if string contains characters outside allowed Latin-ish ranges
    if not ALLOWED_RE.match(s):
        return False

    # Require at least some letters (avoid strings that are mostly symbols)
    alpha_count = sum(ch.isalpha() for ch in s)
    if alpha_count < min_alpha:
        return False

    return True


In [ ]:
df = df_2009

# Strip whitespace
for col in ["artist_name", "track_name"]:
    df[col] = df[col].astype(str).str.strip()

# Apply readability filter
mask = (
    df["artist_name"].apply(is_clean_latin_text) &
    df["track_name"].apply(is_clean_latin_text)
)

df_clean = df[mask].reset_index(drop=True)

print("Rows before:", len(df))
print("Rows after cleaning:", len(df_clean))

out_path = "/content/first_1.3m_rows_2009_clean.csv"
df_clean.to_csv(out_path, index=False)

print("Saved cleaned file to:", out_path)



Rows before: 160372
Rows after cleaning: 155721
Saved cleaned file to: /content/first_1.3m_rows_2009_clean.csv


In [95]:
# Paths to your CSV files
path1 = "/content/first_1.3m_rows_2009_clean.csv"
path2 = "/content/first_1.8m_rows_2009_clean_65+.csv"

# Load CSVs
df1 = pd.read_csv(path1)
df2 = pd.read_csv(path2)

# Concatenate (stack rows)
df_combined = pd.concat([df1, df2], ignore_index=True)

print("Rows after concat:", len(df_combined))
print("Unique users:", df_combined["user_id"].nunique())

# Save combined CSV
out_path = "/content/users_tracks.csv"
df_combined.to_csv(out_path, index=False)

print("Saved combined file to:", out_path)

Rows after concat: 233308
Unique users: 83
Saved combined file to: /content/users_tracks.csv


In [96]:

# Load your dataset
path = "/content/users_tracks.csv"   # change if it's in Drive or named slightly differently
df = pd.read_csv(path)

# --- Parse timestamps ---
df["timestamp_dt"] = pd.to_datetime(df["timestamp"], utc=True, errors="coerce")

# Keep only valid timestamps
df = df.dropna(subset=["timestamp_dt"]).copy()

# Keep only year 2009 (since that's your design)
df = df[df["timestamp_dt"].dt.year == 2009].copy()

# Optional: strip whitespace
for col in ["user_id", "artist_name", "track_name"]:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()

# --- Define windows anchored at start of 2009 ---
start_2009 = pd.Timestamp("2009-01-01", tz="UTC")
windows = [30, 90, 180, 365]

all_results = []

for w in windows:
    window_end = start_2009 + pd.Timedelta(days=w)
    df_w = df[(df["timestamp_dt"] >= start_2009) & (df["timestamp_dt"] < window_end)].copy()
    if df_w.empty:
        continue

    # 1) top artists per user in this window
    artist_counts = (
        df_w.groupby(["user_id", "artist_name"])
            .size()
            .reset_index(name="artist_play_count")
    )

    top_artists = (
        artist_counts.sort_values(["user_id", "artist_play_count"], ascending=[True, False])
                     .groupby("user_id")
                     .head(15)
    )

    # 2) restrict to those artists and find top 1 track per artist (per user)
    df_top_artists = df_w.merge(
        top_artists[["user_id", "artist_name"]],
        on=["user_id", "artist_name"],
        how="inner"
    )

    track_counts = (
        df_top_artists.groupby(["user_id", "artist_name", "track_name"])
                      .size()
                      .reset_index(name="track_play_count")
    )

    top_track_per_artist = (
        track_counts.sort_values(
            ["user_id", "artist_name", "track_play_count", "track_name"],
            ascending=[True, True, False, True]
        )
        .groupby(["user_id", "artist_name"])
        .head(1)
    )

    # Attach artist counts so we keep the top-15 artists ordering
    top_track_per_artist = top_track_per_artist.merge(
        artist_counts, on=["user_id", "artist_name"], how="left"
    )

    # Keep up to 15 rows per user
    final_15 = (
        top_track_per_artist.sort_values(["user_id", "artist_play_count"], ascending=[True, False])
                            .groupby("user_id")
                            .head(15)
                            .reset_index(drop=True)
    )

    # Add window metadata
    final_15["time_window_days"] = w
    final_15["window_start"] = start_2009.isoformat()
    final_15["window_end"] = window_end.isoformat()

    all_results.append(final_15)

# Combine all windows
out = pd.concat(all_results, ignore_index=True) if all_results else pd.DataFrame()

# Save
out_path = "/content/users_tracks_top15_per_window_2009start.csv"
out.to_csv(out_path, index=False)

print("Saved to:", out_path)
print("Users:", out["user_id"].nunique() if not out.empty else 0)
print("Rows:", len(out))

Saved to: /content/users_tracks_top15_per_window_2009start.csv
Users: 83
Rows: 3678


In [97]:
# Load the per-window top15 output
path = "/content/users_tracks_top15_per_window_2009start.csv"
df = pd.read_csv(path)

windows = [30, 90, 180, 365]

# Count rows per (user, window)
counts = (
    df.groupby(["user_id", "time_window_days"])
      .size()
      .reset_index(name="n_rows")
)

# Keep only users that have exactly 15 rows in EVERY window
good_users = (
    counts[counts["time_window_days"].isin(windows)]
    .groupby("user_id")["n_rows"]
    .apply(lambda s: (len(s) == len(windows)) and (s.eq(15).all()))
)

good_users = good_users[good_users].index

# Filter the main dataframe
df_strict = df[df["user_id"].isin(good_users)].reset_index(drop=True)

print("Users before:", df["user_id"].nunique())
print("Users after (exactly 15 per window):", df_strict["user_id"].nunique())
print("Rows after:", len(df_strict))

# Save
out_path = "/content/50_users_tracks_top15_per_window_2009start_STRICT.csv"
df_strict.to_csv(out_path, index=False)
print("Saved to:", out_path)


Users before: 83
Users after (exactly 15 per window): 50
Rows after: 3000
Saved to: /content/50_users_tracks_top15_per_window_2009start_STRICT.csv
